<a href="https://colab.research.google.com/github/MuhammadHakami/NeuroAi-EmbodiedAI/blob/main/notebooks/qualitative_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Watch any trained controller reach — qualitative check

*Click the badge to run this notebook in Google Colab — the setup cell below installs
everything and fetches trained weights on demand. GPU runtime recommended.*


In [ ]:
# @title Setup (Google Colab) — run me first  { display-mode: "form" }
# --- capstone-colab-bootstrap ---------------------------------------------------------
# Makes this notebook runnable on a fresh Colab VM. On a local machine every
# branch below is skipped, so the cell is a no-op and safe to keep at the top.
import os, subprocess, sys

def _on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

REPO_URL = "https://github.com/MuhammadHakami/NeuroAi-EmbodiedAI.git"
REPO_DIR = "/content/NeuroAI_EmbodiedAi_Capstone"

if _on_colab():
    if not os.path.isdir(REPO_DIR):
        # --recurse-submodules pulls MotorNet (pinned v0.3.0) and nlb_tools with it
        subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "-e", os.path.join(REPO_DIR, "MotorNet")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "numpy", "scipy", "scikit-learn", "matplotlib"], check=True)

    # Trained weights / neural datasets are NOT in git (see README "Data and weights").
    # Mount Drive if you want to reuse or save checkpoints; training works without it.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE = "/content/drive/MyDrive/neuroai_capstone"
        os.makedirs(DRIVE, exist_ok=True)
        os.environ["CAPSTONE_DRIVE"] = DRIVE
        print("Drive ready ->", DRIVE)
    except Exception as e:
        print("Drive not mounted (training still works, checkpoints stay on the VM):", e)

# `notebooks/` must be importable for motor_core / motor_zoo / plausible_learners
sys.path.insert(0, os.getcwd())

import torch as th
print("torch", th.__version__, "| CUDA", th.cuda.is_available(),
      "|", th.cuda.get_device_name(0) if th.cuda.is_available() else "CPU")
if not th.cuda.is_available():
    print("WARNING: no GPU. Colab: Runtime > Change runtime type > T4 GPU.")


In [ ]:
# ==============================================================================
# Qualitative analysis -- watch ANY trained controller solve the reach
# ------------------------------------------------------------------------------
# Self-contained. Weights are fetched from Google Drive automatically if they are
# not already in notebooks/save/models.
#
#     run_model("save/models/kinesis.pt")                        # 4 runs, 1.0 kg
#     run_model("save/models/shac.pt", n_runs=6, ball_weight=2.1)          # one weight
#     run_model("save/models/eprop.pt", ball_weight=[0.5, 1.2, 2.1, 2.5])  # several
#     run_model("save/models/btsp.pt", render=False)                       # metrics only
#     run_all()                                                            # every model
# ==============================================================================
import os, sys
sys.path.insert(0, os.getcwd())
from weights import ensure_weights
ensure_weights()          # no-op when the checkpoints are already there

# ==============================================================================
# X. Watch a controller reach -- modular episode video with the body drawn in
# ------------------------------------------------------------------------------
# render_runs(model, n_runs, ball_weight) rolls out `n_runs` independent EPISODES of ANY
# learner and renders them as one video. MotorNet's ReluPointMass24 is a SINGLE point mass
# per episode (its joint state == its fingertip -- there is no elbow/shoulder), driven by
# 4 muscles that run from the fixed corner anchors to the point mass. So the "body" we draw
# each frame, per run, is: the fingertip (the point mass / joint), the 4 muscle paths
# (anchor -> fingertip, thickened + brightened by that muscle's activation), the fixed
# anchors, and the reach trail. Every run is drawn TRANSLUCENT so several episodes overlay
# without overbearing the view -- active muscles pop, the rest fade.
#   * n_runs      -- number of episodes (independent reaches, each its own target).
#   * ball_weight -- point-mass weight (kg): scalar (same ball each run) or list of length
#                    n_runs. Default 1.0 = the trained mass, so every reach succeeds.
# ==============================================================================
import os, glob, numpy as np, torch as th
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib import cm
from IPython.display import Video, Image
import motornet as mn
# SELF-CONTAINED: this cell needs no other cell to have been run. Restart the kernel and
# execute just this one -- every symbol it uses is imported here.
from motor_zoo import (DEVICE, ReachEnv, env_to, MODEL_DIR, make_mass_env, eval_metrics,
                       count_params, updates_per_episode, TRAIN_MASSES, VAL_MASSES,
                       MotorNetRef, BPTTGRU, SHAC, SAC, FastTD3, SimbaV2, Kinesis, Dendritron)
from weights import ensure_weights
from plausible_learners import EProp, RTRRL, BTSP, RSTDP, PredictiveCoding, Hebb3

def _env_with_mass(mass):
    return env_to(ReachEnv(effector=mn.effector.ReluPointMass24(mass=float(mass)), max_ep_duration=1.), DEVICE)

def render_runs(model, n_runs=4, ball_weight=1.0, out=os.path.join("save", "episode.mp4"),
                fps=20, title=None, seed=0):
    weights = list(ball_weight) if isinstance(ball_weight, (list, tuple, np.ndarray)) else [ball_weight] * n_runs
    assert len(weights) == n_runs, f"ball_weight must be a scalar or a list of length n_runs={n_runs}, got {len(weights)}"

    runs = []
    for i, w in enumerate(weights):
        env = _env_with_mass(w); n = int(env.max_ep_duration / env.dt)
        th.manual_seed(seed + i)                                     # a different random target each episode
        obs, info = env.reset(options={"batch_size": 1}); h = model.init_state(1)
        TIP, GOAL, ACT = [], [], []
        with th.no_grad():
            for t in range(n):
                a, h = model.act(obs, h)
                TIP.append(np.asarray(obs[0, 2:4].detach().cpu())); GOAL.append(np.asarray(obs[0, 0:2].detach().cpu()))
                ACT.append(np.asarray(a[0].detach().cpu())); obs, r, term, trunc, info = env.step(a)
        TIP, GOAL = np.array(TIP), np.array(GOAL)
        runs.append(dict(w=w, tip=TIP, goal=GOAL, act=np.array(ACT), err=np.linalg.norm(TIP - GOAL, axis=1) * 100))
    _pc = env.effector._path_coordinates[0, :, 0::2].T
    anchors = np.asarray(_pc.detach().cpu() if hasattr(_pc, "detach") else _pc)   # 4 fixed muscle origins
    n = len(runs[0]["tip"]); vary = len(set(weights)) > 1
    col = [cm.tab10(i % 10) for i in range(n_runs)]
    HAZE = min(0.9, 0.55 + 0.12 * n_runs)                            # more runs -> fainter, to stay readable

    fig, ax = plt.subplots(figsize=(6.8, 6.4)); lim = float(np.abs(anchors).max()) + 0.5
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.axis("off")
    # fixed body scaffold: the 4 muscle anchors, drawn once, faint
    ax.scatter(anchors[:, 0], anchors[:, 1], s=110, facecolors="none", edgecolors="#2F3E46", lw=1.4, zorder=2)
    for A in anchors: ax.annotate("", xy=(A[0]*.92, A[1]*.92), xytext=(0, 0), zorder=1,
                                  arrowprops=dict(arrowstyle="-", color="#B0B7C3", lw=0.6, alpha=0.4))
    muscles, trails, dots, halos = [], [], [], []
    for i, r in enumerate(runs):
        ax.plot([r["goal"][-1, 0]], [r["goal"][-1, 1]], marker="*", ms=20, c=col[i], ls="", alpha=0.9, zorder=6)
        muscles.append([ax.plot([], [], "-", c=col[i], solid_capstyle="round", zorder=4)[0] for _ in range(4)])
        halos.append(ax.plot([], [], "o", c=col[i], mfc="none", zorder=4)[0])          # co-contraction / stiffness halo
        trails.append(ax.plot([], [], "-", c=col[i], lw=1.6, alpha=0.35 * HAZE, zorder=3)[0])
        lab = f"{r['w']:g} kg" if vary else f"reach {i+1}"
        dots.append(ax.plot([], [], "o", ms=12, c=col[i], alpha=0.95, zorder=7, label=lab)[0])
    txt = ax.text(-lim + 0.1, lim - 0.28, "", fontsize=11, fontweight="bold", color="#1F2933")
    ax.set_title(f"{title or getattr(model, 'name', 'controller')} — {n_runs} reach{'es' if n_runs>1 else ''}"
                 "\nfingertip + 4 muscles (thickness = activation); ★ = each episode's target", fontsize=11.5, fontweight="bold")
    ax.legend(loc="lower right", title=("ball weight" if vary else "episode"), fontsize=8, title_fontsize=8, framealpha=.9)

    def update(f):
        arts = [txt]
        for i, r in enumerate(runs):
            P = r["tip"][f]; a4 = r["act"][f]
            for m in range(4):
                muscles[i][m].set_data([anchors[m, 0], P[0]], [anchors[m, 1], P[1]])
                muscles[i][m].set_alpha((0.10 + 0.75 * float(a4[m])) * HAZE)            # inactive fades, active pops
                muscles[i][m].set_linewidth(0.8 + 6 * float(a4[m]))
            cc = float(np.minimum(a4[0], a4[3]) + np.minimum(a4[1], a4[2]))            # co-contraction proxy
            halos[i].set_data([P[0]], [P[1]]); halos[i].set_markersize(14 + 34 * cc); halos[i].set_alpha(0.25 * HAZE)
            trails[i].set_data(r["tip"][:f + 1, 0], r["tip"][:f + 1, 1]); dots[i].set_data([P[0]], [P[1]])
            arts += muscles[i] + [halos[i], trails[i], dots[i]]
        txt.set_text(f"t = {f * 10:3d} ms   mean error = {np.mean([r['err'][f] for r in runs]):.1f} cm")
        return arts

    ani = FuncAnimation(fig, update, frames=n, interval=1000.0 / fps, blit=False)
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    try:
        ani.save(out, writer=FFMpegWriter(fps=fps, bitrate=2600))
    except Exception:
        out = os.path.splitext(out)[0] + ".gif"; ani.save(out, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f"saved {out}  |  {n_runs} episode(s): " + ", ".join(f"{r['w']:g}kg→{r['err'][-1]:.1f}cm" for r in runs))
    return out

# ---- MODULAR: point it at a weights file and it does the rest -------------------------
#
#     run_model("save/models/kinesis.pt")                       # defaults: 4 runs, 1.0 kg
#     run_model("save/models/shac.pt", n_runs=6, ball_weight=2.1)         # ONE weight
#     run_model("save/models/eprop.pt", ball_weight=[0.5, 1.2, 2.1, 2.5]) # a LIST of weights
#     run_model("save/models/btsp.pt", render=False)                      # metrics only
#
# The class is inferred from the filename, so `kinesis.pt` rebuilds a Kinesis. Pass
# `model=SomeClass` to override, and any extra kwargs go to that class's constructor.
#
ZOO = {"motornet_ref": MotorNetRef, "bptt_gru": BPTTGRU, "shac": SHAC, "sac": SAC,
       "fasttd3": FastTD3, "simbav2": SimbaV2, "eprop": EProp, "rtrrl": RTRRL, "btsp": BTSP,
       "kinesis": Kinesis, "rstdp": RSTDP, "predcode": PredictiveCoding, "hebb3": Hebb3,
       "dendritron": Dendritron}


def run_model(weights_path, n_runs=4, ball_weight=1.0, model=None, render=True, **ctor_kwargs):
    """Load a checkpoint, score it on the held-out balls, and show it reaching.

    weights_path : path to a .pt (the class is inferred from the file stem unless `model=` is given)
    n_runs       : how many independent episodes to overlay in the video
    ball_weight  : a single number (2.1) OR a list ([0.5, 1.2, 2.1]) -- both accepted
    render       : False to skip the video and just return the metrics

    Returns a dict of metrics; the video is displayed automatically.
    """
    weights_path = os.fspath(weights_path)
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"no checkpoint at {weights_path!r}")
    tag = os.path.splitext(os.path.basename(weights_path))[0]

    cls = model or ZOO.get(tag)
    if cls is None:
        raise KeyError(f"cannot infer a model class from {tag!r}. "
                       f"Pass model=<Class>, or name the file one of: {sorted(ZOO)}")

    # a single number and a list are both valid -- normalise to a list of floats
    weights = [float(ball_weight)] if np.isscalar(ball_weight) else [float(w) for w in ball_weight]
    if n_runs is None:
        n_runs = len(weights)

    m = cls(make_mass_env(DEVICE, TRAIN_MASSES), **ctor_kwargs)
    sd = th.load(weights_path, map_location=DEVICE)
    m.load_state_dict(sd.get("state_dict", sd) if isinstance(sd, dict) else sd, strict=False)

    mt = eval_metrics(make_mass_env(DEVICE, VAL_MASSES), m)
    pol, aux = count_params(m)
    out = dict(tag=tag, name=getattr(m, "name", tag), kind=getattr(m, "kind", "?"),
               weights=weights_path, err_cm=mt["err_cm"], completion=mt["completion"],
               completion2=mt["completion2"], params_policy=pol, params_aux=aux,
               updates_per_episode=updates_per_episode(m, make_mass_env(DEVICE, VAL_MASSES)),
               ball_weights=weights, n_runs=n_runs, model=m)

    print(f"{out['name']}   [{out['kind']}]\n"
          f"  weights            {weights_path}\n"
          f"  held-out error     {out['err_cm']:.2f} cm   (balls {VAL_MASSES} kg, never trained on)\n"
          f"  completion         {out['completion']:.1f}% @5cm   {out['completion2']:.1f}% @2cm\n"
          f"  params             {pol:,} policy  +  {aux:,} auxiliary\n"
          f"  updates/episode    {out['updates_per_episode']:.4g}\n"
          f"  rendering          {n_runs} run(s) at {weights} kg")

    if render:
        p = render_runs(m, n_runs=n_runs, ball_weight=weights,
                        out=os.path.join("save", f"episode_{tag}.mp4"),
                        title=f"{out['name']} - {', '.join(f'{w:g}kg' for w in weights)}")
        out["video"] = p
        display(Video(p, embed=True) if p.endswith(".mp4") else Image(p))
    return out


def run_all(n_runs=4, ball_weight=(0.5, 1.2, 2.1, 2.5), render=False):
    """Every checkpoint in MODEL_DIR, scored into one table."""
    rows = []
    for p in sorted(glob.glob(os.path.join(MODEL_DIR, "*.pt"))):
        try:
            rows.append(run_model(p, n_runs=n_runs, ball_weight=ball_weight, render=render))
        except Exception as e:
            print(f"{os.path.basename(p):<18} FAILED: {type(e).__name__}: {e}")
    rows.sort(key=lambda r: -r["completion"])
    print(f"\n{'model':<34}{'err cm':>8}{'compl%':>8}{'params':>10}{'upd/ep':>9}")
    for r in rows:
        print(f"{r['name'][:33]:<34}{r['err_cm']:>8.2f}{r['completion']:>8.1f}"
              f"{r['params_policy']:>10,}{r['updates_per_episode']:>9.4g}")
    return rows


ensure_weights()   # downloads + unzips the model zoo only if save/models is empty

# ---- change this line to inspect any other model -------------------------------------
_ = run_model("save/models/kinesis.pt", n_runs=4, ball_weight=[0.6, 1.0, 1.3, 1.7])
